# Unsupervised Learning

In [ ]:
from sklearn.cluster import KMeans
from met_cleaning import clean_met_data
from sklearn.model_selection import train_test_split
# from sklearn.metrics import confusion_matrix
# from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
# from imblearn.over_sampling import SMOTE

In [2]:
df = clean_met_data()

/Users/mollystark/Desktop/michigan/milestone 2/SIADS-Milestone-II/src/notebooks/met_cleaning.py:20: DtypeWarning: Columns (5,7,10,11,12,13,14,34,35,36,37,38,39,40,41,42,43,44,45,46) have mixed types. Specify dtype option on import or set low_memory=False.
  met = pd.read_csv('https://media.githubusercontent.com/media/metmuseum'


In [3]:
X = df.drop('is_significant', axis=1)
y = df['is_significant']

In [4]:
numeric_features = ['access_year', 'extracted_date']
numeric_transformer = Pipeline(steps=[("imputer",
                                       SimpleImputer(strategy="constant", fill_value=0)),])

In [5]:
categorical_features = ['object_name','culture','portfolio', #'title','department',
                        'artist_display_name','artist_nationality','mapped_country','medium', 'artist_gender']
categorical_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy="constant", 
                                                                    fill_value="missing")),
                                      ('onehot', OneHotEncoder(handle_unknown="ignore"))])

In [6]:
tag_features = ['tags']
tag_transformer = Pipeline(steps=[('tfidf', TfidfVectorizer())])

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
        # ("tags", tag_transformer, tag_features)
    ]
)

In [8]:
X = preprocessor.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.8,random_state=42)

In [9]:
clf = KMeans(n_clusters=5, random_state=42, n_init="auto").fit(X)

/Users/mollystark/Desktop/michigan/milestone 2/SIADS-Milestone-II/.venv/lib/python3.13/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: divide by zero encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/mollystark/Desktop/michigan/milestone 2/SIADS-Milestone-II/.venv/lib/python3.13/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: overflow encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/mollystark/Desktop/michigan/milestone 2/SIADS-Milestone-II/.venv/lib/python3.13/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: invalid value encountered in matmul
  current_pot = closest_dist_sq @ sample_weight


In [11]:
clf.get_feature_names_out

<bound method ClassNamePrefixFeaturesOutMixin.get_feature_names_out of KMeans(n_clusters=5, random_state=42)>

In [10]:
metrics = {
    "train_data":{
        "score": clf.score(X_train, y_train),
        "mae": mean_absolute_error(y_train, clf.predict(X_train))
    },
    "test_data":{
        "score": clf.score(X_test, y_test),
        "mae": mean_absolute_error(y_test, clf.predict(X_test))
    }
}

print(metrics)


{'train_data': {'score': -1718575780.8414795, 'mae': 0.6464723531049272}, 'test_data': {'score': -6510792484.970271, 'mae': 0.6424754810356604}}
